In [1]:
import tensorflow as tf
import numpy as np
import pandas as pd
import gymnasium as gym
from gymnasium import spaces
from pathlib import Path
import birja
from keras import layers as lay
import keras
import matplotlib.pyplot as plt
import random



In [2]:
import tensorflow as tf
import numpy as np
import pandas as pd
import gymnasium as gym
from pathlib import Path
import birja

tf.config.threading.set_inter_op_parallelism_threads(1)
tf.config.threading.set_intra_op_parallelism_threads(1)

gym.envs.registration.register(
    id='TradingEnv_2_v0',
    entry_point='birja:Trading_3',
    max_episode_steps=10000000000)
#return {'window_data':column_stak,'action_one_hot':dirat,'scalars':scalar}
env = gym.make('TradingEnv_2_v0')


In [3]:
num_action=env.action_space.n
gamma=0.99
lam=0.95

print(hasattr(env,'end_episode'))
env.unwrapped.current_step+=random.randint(80000,115000)
#env.end_episode(2)

False


In [4]:
#@tf.py_function(Tout=[tf.float32,tf.bool])
def end_episode(reward):
            
            env_ = env.unwrapped
            reward=reward
            close=env_.data["close"][env_.current_step+env_.window-2]
            #d_p_l=((close-env_.prise)/env_.prise)*env_.direction if env_.prise!=0 else 0
            d_p_l=env_.unrealized_pnl
            #print('dpl:',d_p_l)
            #print("step_in_episode:",env_.step_in_episode)
            #print("rewerd0:",reward)
            if d_p_l>0:
                 reward+=d_p_l*12*10
            elif d_p_l<0:
                  reward+=d_p_l*12*5
            #print("close:",close)
            reward=tf.tanh(reward/3.0)

                            # штраф за выход из позиции
            reward+=-1.5 if -0.0002>=env_.balance-env_.balance_start>=0.0002 else 0
            
            #print("rewerd2:",reward)
            
            if env_.prise!=0:
                env_.balance+=(((close-env_.prise)/env_.prise)*env_.direction)*env_.position*12-0.00052*env_.position*12 - 0.00052*(((close-env_.prise)/env_.prise)*env_.direction+1)*env_.position*12
            total_balans=(env_.balance-env_.balance_start)
            reward+=total_balans
            
            end_data=tf.constant([False,],tf.bool)
            if (env_.len_data-env_.window-1)-env_.current_step<672:
                end_data=tf.constant([True,],tf.bool)
            tf.print('done:',(env_.len_data-env_.window-1)-env_.current_step<672,'| trun:',env_.step_in_episode>=env_.step_per_episode)
            tf.print("balans:",env_.balance,'|'"trade:",env_.trade_count,'|',"env step:",env_.current_step,'|','step in episode:',env_.step_in_episode)

                  
            env_.position=0
            env_.prise=0
            env_.direction=0
            env_.unrealized_pnl=0
            env_.time_in_trade=0
            env_.step_in_episode=0
            env_.trade_count=0
            env_.balance=env_.balance_start
            
            return tf.constant(reward,tf.float32),end_data

In [5]:
#@tf.py_function(Tout=[tf.float32,tf.float32,tf.float32,tf.float32,tf.float32,tf.bool])
def step(action):
    observation,reward__,done,trun,info=env.step(int(action))
    #{'window_data':column_stak,'action_one_hot':dirat,'scalars':scalar}
    observation=[tf.reshape(tf.constant(observation['window_data'],tf.float32),(288,10,)),
                 tf.reshape(tf.constant(observation['action_one_hot'],tf.float32),(3,)),
                 tf.reshape(tf.constant(observation['scalars'],tf.float32),(6,))]
    reward__=tf.tanh(tf.constant(reward__,tf.float32)/3.0)
    done=done or trun
    done=tf.constant(float(done),tf.float32)
    trun=tf.constant(bool(done),tf.bool)
    return observation[0],observation[1],observation[2],reward__,done,trun,info['rt']

#@tf.py_function(Tout=[tf.float32,tf.float32,tf.float32])
def reset():
    obs,_=env.reset()
    env.unwrapped.current_step+=random.randint(80000,115000)
    
    obs=[tf.reshape(tf.constant(obs['window_data'],tf.float32),(288,10,)),
                 tf.reshape(tf.constant(obs['action_one_hot'],tf.float32),(3,)),
                 tf.reshape(tf.constant(obs['scalars'],tf.float32),(6,))]
    return obs[0],obs[1],obs[2]


@tf.function
def get_discounted_sum(x,gamma):
    x=tf.cast(x[::-1],tf.float32)
    gamma=tf.constant(gamma,tf.float32)
    y=tf.TensorArray(tf.float32,0,dynamic_size=True)
    y0=tf.constant(0.0)
    y0_shape=y0.shape

    for i in tf.range(tf.shape(x)[0]):
        y0=x[i]+y0*gamma
        y0=tf.reshape(y0,y0_shape)
        y=y.write(i,y0)

    return y.stack()[::-1]

@tf.function
def log_probability_get(logits,action):
    #log=tf.math.log_softmax(logits)
    log=tf.nn.log_softmax(logits)
    return tf.reduce_sum(tf.one_hot(tf.cast(action,tf.int32),num_action)*log,-1)


In [6]:
@tf.function
def positional_encoding(seq_len, dim):
    positions = tf.range(seq_len)[:, tf.newaxis]
    dims = tf.range(dim)[tf.newaxis, :]
    angle_rates = 1 / tf.pow(10000., (2 * (dims//2)) / tf.cast(dim, tf.float32))
    angle_rads = tf.cast(positions, tf.float32) * angle_rates
    sines = tf.sin(angle_rads[:, 0::2])
    cosines = tf.cos(angle_rads[:, 1::2])
    pos_encoding = tf.concat([sines, cosines], axis=-1)
    return pos_encoding

observations_0,observations_1,observations_2=reset()
#{'window_data':column_stak,'action_one_hot':dirat,'scalars':scalar}
input_data=lay.Input((288,10),1,name="window_data")
input_action=lay.Input((3,),1,name='action_one_hot')
input_scalar=lay.Input((6,),1,name='scalars')

#Хочешь, я покажу архитектуру PPO с Conv1D + Attention (вход — 50 свечей, выход — действие + value), с объяснением, как именно данные идут через модель?
#Это как раз то, что используют в продвинутых трейдинговых RL-моделях.
#padding="causa

x=lay.Conv1D(128,3,padding="causal",activation='relu')(input_data)
x=lay.Conv1D(128,5,padding="causal",activation='relu')(x)

pos_encoding=positional_encoding(288.,128.)




x = x + pos_encoding

x=lay.LayerNormalization()(x)
x1=lay.MultiHeadAttention(4,32,dropout=0.1)(x,x)
x=lay.Add()([x,lay.Conv1D(128,1,activation='relu')(x1)])
x=lay.LayerNormalization()(x)
x1=lay.Conv1D(128,3,activation='relu',padding='causal')(x)
x=lay.Add()([x,x1])
x=lay.LayerNormalization()(x)
x=lay.GlobalAvgPool1D()(x)


x1=lay.Dense(32,'relu')(input_action)
x2=lay.Dense(64,'relu',kernel_regularizer=keras.regularizers.L2())(input_scalar)

x2=lay.concatenate([x,x1,x2])
x2=lay.Dropout(0.15)(x2)
x=lay.Dense(256,'relu',kernel_regularizer=keras.regularizers.L2())(x2)
x=lay.Dense(64,'relu',kernel_regularizer=keras.regularizers.L2())(x)
y=lay.Dense(4)(x)

x1=lay.Dense(128,'relu',kernel_regularizer=keras.regularizers.L2())(x2)
x1=lay.Dense(32,'relu')(x1)
y1=lay.Dense(1)(x1)
actor=keras.Model([input_data,input_action,input_scalar],y)
critic=keras.Model([input_data,input_action,input_scalar],y1)
#-------------------------------------------------
#-------------------------------------------------

#actor.load_weights('model_actor_1.weights.h5')
#critic.load_weights('model_critic_1.weights.h5')

#actor.summary()
#critic.summary()


d:\Python\Lib\site-packages\gymnasium\utils\passive_env_checker.py:134: UserWarning: WARN: The obs returned by the `reset()` method was expecting numpy array dtype to be float32, actual type: float64
  logger.warn(
d:\Python\Lib\site-packages\gymnasium\utils\passive_env_checker.py:158: UserWarning: WARN: The obs returned by the `reset()` method is not within the observation space.
  logger.warn(f"{pre} is not within the observation space.")


In [7]:
batch_dim=1
DATA_SHAPE=(288,10)
ACTIONS_SHAPE=(3,)
SCAL_SHAPE=(6,)

observation_data,observation_action,observation_scal=reset()
observation_data,observation_action,observation_scal=tf.cast(tf.expand_dims(tf.reshape(tf.squeeze(observation_data),DATA_SHAPE),0),tf.float32),tf.cast(tf.expand_dims(tf.reshape(tf.squeeze(observation_action),ACTIONS_SHAPE),0),tf.float32),tf.cast(tf.expand_dims(tf.reshape(tf.squeeze(observation_scal),SCAL_SHAPE),0),tf.float32)

_ = actor({'window_data':observation_data,'action_one_hot':observation_action,'scalars':observation_scal})
_ = critic({'window_data':observation_data,'action_one_hot':observation_action,'scalars':observation_scal})

a=''
actions=[]
#actor.load_weights("D:\model_actor_3.weights (4).h5")
index_op=[]
index_close=[]
observation_data,observation_action,observation_scal=reset()
actor.load_weights(Path("D:\\model_actor_6.weights (2).h5"),by_name=True)
critic.load_weights(Path("D:\\model_critic_6.weights (2).h5"),by_name=True)
dirs=[]

for i in range(100):
    observation_data,observation_action,observation_scal=tf.cast(tf.expand_dims(tf.reshape(tf.squeeze(observation_data),DATA_SHAPE),0),tf.float32),tf.cast(tf.expand_dims(tf.reshape(tf.squeeze(observation_action),ACTIONS_SHAPE),0),tf.float32),tf.cast(tf.expand_dims(tf.reshape(tf.squeeze(observation_scal),SCAL_SHAPE),0),tf.float32)
    logit=actor({'window_data':observation_data,'action_one_hot':observation_action,'scalars':observation_scal},)
    action = tf.argmax(logit,-1)
       # shape: (1,)
    action = int(action.numpy()[0])
    actions.append(action)
    #print(logit.numpy().tolist())
    #print(action)      # превращаем в обычный int
    #action=tf.random.uniform((),0,3,tf.int32)
    #action=1

    observation_data,observation_action,observation_scal,reward,done,logical_bool,rt=step(int(action))
    env__=env.unwrapped
    if rt==1:
        index_op.append(i-100+100)
        dirs.append(env__.direction)
    elif rt==-1:
        index_close.append(i-100+100)
    a+=f"action:{action} | reward:{reward:<5.5f} | diration:{env__.direction} | d_p_l:{env__.unrealized_pnl:<3.7f} | prise:{env__.prise:<3.5f} | close -prise:{env__.data["close"][env__.current_step+env__.window-2]-env__.prise}\n"
    #a+=f'step:{i:<5} | action:{action:<10} | reward:{reward:<8.5f} | balans:{env__.balance:<10.5f} | time_in trade:{env__.time_in_trade:<4} | close:{env__.data["close"][env__.current_step+env__.window-2]:<10.5f} | prise:{env__.prise:<10.5f} | delta:{(env__.data["close"][env__.current_step+env__.window-2]-env__.prise)/env__.prise*env__.direction*12:<5.5f} | diration:{env__.direction:<10} | upl:{env__.unrealized_pnl:<10.5f} komisiy open:{-0.00052*env__.position*12} | komisy close:{- 0.00052*(((env__.data["close"][env__.current_step+env__.window-2]-env__.prise)/env__.prise)*env__.direction+1)*env__.position*12} observation_data:{observation_data[0]} | observation_action:{observation_action} | observation_scal:{observation_scal}\n'
    #a+=f'obs_data:{observation_data.numpy().tolist():<5.10f}'+'\n'
    p=pd.DataFrame(observation_data.numpy().tolist()[-100:],columns=['open','high','low','close','volue','rsi','adx','bb','obv','atr'])
reward1,end=end_episode(reward)
env__=env.unwrapped
a+='-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------\n'
a+=f"reward:{reward1:<8.5f} | end:{end} | rewaward_orig:{reward:<8.5f} | close:{env__.data["close"][env__.current_step+env__.window-2]:<5.5f} | D_P_L:{((env__.data["close"][env__.current_step+env__.window-2]-env__.prise))*env__.direction if env__.prise!=0 else 0:<2.5f} | prise:{env__.prise:<5.3f} | done:{(env__.len_data-env__.window-1)-env__.current_step<672:<1.1f} "
'''Никита, [05.11.2025 14:41]
взаимосвяз данных и вычеслений
''' 
with open('analitik.txt','w') as f:
    f.write(a)
for i in actions:
    if i==0:
        continue
    print(i,end=' ')
print()
import mplfinance as mpf

df = p

import numpy as np
import mplfinance as mpf

df = p.reset_index(drop=True)
df.index = pd.to_datetime(df.index, unit='s')

opens_y  = np.full(len(df), np.nan)
closes_y = np.full(len(df), np.nan)
opens_label = [""]*len(df)
closes_label = [""]*len(df)
#print(dirs)
# --- пример: dirs для открытия/закрытия ---
dirs_open = dirs  # направления открытия
dirs_close = dirs  # если для закрытия отдельный список, используй свой
'''
# --- метки открытия ---
for idx, dir_ in zip(index_op, dirs_open):
    if idx is None or not (0 <= idx < len(df)):
        continue
    opens_y[idx] = df['low'][idx] * 0.993   # точка немного ниже свечи
    opens_label[idx] = f"Dir:{dir_}"

# --- метки закрытия ---
for idx, dir_ in zip(index_close, dirs_close):
    if idx is None or not (0 <= idx < len(df)):
        continue
    closes_y[idx] = df['high'][idx] * 1.007  # точка немного выше свечи
    closes_label[idx] = f"CLOSE\nDir:{dir_}"

# --- строим addplot для точек ---
ap = []
if np.any(~np.isnan(opens_y)):
    ap.append(mpf.make_addplot(opens_y, scatter=True, marker='^', markersize=35, color='green', alpha=0.8))
if np.any(~np.isnan(closes_y)):
    ap.append(mpf.make_addplot(closes_y, scatter=True, marker='v', markersize=35, color='red', alpha=0.8))

fig, axlist = mpf.plot(df, type='candle', style='charles', addplot=ap, returnfig=True)
ax = axlist[0]  # основной график

# --- текст над/под точкой ---
delta = 0.1 * (df['high'].max() - df['low'].min())  # 2% диапазона графика

for i, label in enumerate(opens_label):
    if label != "":
        ax.text(i, df['low'][i] - delta, label, fontsize=6, ha='center', va='top',
                color='white', bbox=dict(facecolor='green', alpha=0.6, pad=1))

for i, label in enumerate(closes_label):
    if label != "":
        ax.text(i, df['high'][i] + delta, label, fontsize=6, ha='center', va='bottom',
                color='white', bbox=dict(facecolor='red', alpha=0.6, pad=1))
57286'''

d:\Python\Lib\site-packages\gymnasium\utils\passive_env_checker.py:134: UserWarning: WARN: The obs returned by the `step()` method was expecting numpy array dtype to be float32, actual type: float64
  logger.warn(
d:\Python\Lib\site-packages\gymnasium\utils\passive_env_checker.py:158: UserWarning: WARN: The obs returned by the `step()` method is not within the observation space.
  logger.warn(f"{pre} is not within the observation space.")


done: False | trun: False
balans: 99.66276185562445 |trade: 10 | env step: 193039 | step in episode: 100
2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 


'\n# --- метки открытия ---\nfor idx, dir_ in zip(index_op, dirs_open):\n    if idx is None or not (0 <= idx < len(df)):\n        continue\n    opens_y[idx] = df[\'low\'][idx] * 0.993   # точка немного ниже свечи\n    opens_label[idx] = f"Dir:{dir_}"\n\n# --- метки закрытия ---\nfor idx, dir_ in zip(index_close, dirs_close):\n    if idx is None or not (0 <= idx < len(df)):\n        continue\n    closes_y[idx] = df[\'high\'][idx] * 1.007  # точка немного выше свечи\n    closes_label[idx] = f"CLOSE\nDir:{dir_}"\n\n# --- строим addplot для точек ---\nap = []\nif np.any(~np.isnan(opens_y)):\n    ap.append(mpf.make_addplot(opens_y, scatter=True, marker=\'^\', markersize=35, color=\'green\', alpha=0.8))\nif np.any(~np.isnan(closes_y)):\n    ap.append(mpf.make_addplot(closes_y, scatter=True, marker=\'v\', markersize=35, color=\'red\', alpha=0.8))\n\nfig, axlist = mpf.plot(df, type=\'candle\', style=\'charles\', addplot=ap, returnfig=True)\nax = axlist[0]  # основной график\n\n# --- текст над/

In [15]:
batch_dim=1
DATA_SHAPE=(288,10)
ACTIONS_SHAPE=(3,)
SCAL_SHAPE=(6,)
actor=keras.Model([input_data,input_action,input_scalar],y)
critic=keras.Model([input_data,input_action,input_scalar],y1)
observation_data,observation_action,observation_scal=reset()
observation_data,observation_action,observation_scal=tf.cast(tf.expand_dims(tf.reshape(tf.squeeze(observation_data),DATA_SHAPE),0),tf.float32),tf.cast(tf.expand_dims(tf.reshape(tf.squeeze(observation_action),ACTIONS_SHAPE),0),tf.float32),tf.cast(tf.expand_dims(tf.reshape(tf.squeeze(observation_scal),SCAL_SHAPE),0),tf.float32)

_ = actor({'window_data':observation_data,'action_one_hot':observation_action,'scalars':observation_scal})
_ = critic({'window_data':observation_data,'action_one_hot':observation_action,'scalars':observation_scal})

a=''
actions=[]
#actor.load_weights("D:\model_actor_3.weights (4).h5")
index_op=[]
index_close=[]
observation_data,observation_action,observation_scal=reset()
actor.load_weights(Path("D:\\model_actor_6_last.weights (1).h5"),skip_mismatch=True,by_name=True)
critic.load_weights(Path("D:\\model_critic_6_last.weights (1).h5"),skip_mismatch=True,by_name=True)
dirs=[]

for i in range(100):
    observation_data,observation_action,observation_scal=tf.cast(tf.expand_dims(tf.reshape(tf.squeeze(observation_data),DATA_SHAPE),0),tf.float32),tf.cast(tf.expand_dims(tf.reshape(tf.squeeze(observation_action),ACTIONS_SHAPE),0),tf.float32),tf.cast(tf.expand_dims(tf.reshape(tf.squeeze(observation_scal),SCAL_SHAPE),0),tf.float32)
    logit=actor({'window_data':observation_data,'action_one_hot':observation_action,'scalars':observation_scal},)
    
    action = tf.argmax(logit,-1)
       # shape: (1,)
    action = int(action.numpy()[0])
    actions.append(action)
    #print(logit.numpy().tolist())
    #print(action)      # превращаем в обычный int
    #action=tf.random.uniform((),0,3,tf.int32)
    #action=1

    observation_data,observation_action,observation_scal,reward,done,logical_bool,rt=step(int(action))
    env__=env.unwrapped
    if rt==1:
        index_op.append(i-100+100)
        dirs.append(env__.direction)
    elif rt==-1:
        index_close.append(i-100+100)
    a+=f"action:{action} | reward:{reward:<5.5f} | diration:{env__.direction} | d_p_l:{env__.unrealized_pnl:<3.7f} | prise:{env__.prise:<3.5f} | close -prise:{env__.data["close"][env__.current_step+env__.window-2]-env__.prise}\n"
    #a+=f'step:{i:<5} | action:{action:<10} | reward:{reward:<8.5f} | balans:{env__.balance:<10.5f} | time_in trade:{env__.time_in_trade:<4} | close:{env__.data["close"][env__.current_step+env__.window-2]:<10.5f} | prise:{env__.prise:<10.5f} | delta:{(env__.data["close"][env__.current_step+env__.window-2]-env__.prise)/env__.prise*env__.direction*12:<5.5f} | diration:{env__.direction:<10} | upl:{env__.unrealized_pnl:<10.5f} komisiy open:{-0.00052*env__.position*12} | komisy close:{- 0.00052*(((env__.data["close"][env__.current_step+env__.window-2]-env__.prise)/env__.prise)*env__.direction+1)*env__.position*12} observation_data:{observation_data[0]} | observation_action:{observation_action} | observation_scal:{observation_scal}\n'
    #a+=f'obs_data:{observation_data.numpy().tolist():<5.10f}'+'\n'
    p=pd.DataFrame(observation_data.numpy().tolist()[-100:],columns=['open','high','low','close','volue','rsi','adx','bb','obv','atr'])
reward1,end=end_episode(reward)
env__=env.unwrapped
a+='-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------\n'
a+=f"reward:{reward1:<8.5f} | end:{end} | rewaward_orig:{reward:<8.5f} | close:{env__.data["close"][env__.current_step+env__.window-2]:<5.5f} | D_P_L:{((env__.data["close"][env__.current_step+env__.window-2]-env__.prise))*env__.direction if env__.prise!=0 else 0:<2.5f} | prise:{env__.prise:<5.3f} | done:{(env__.len_data-env__.window-1)-env__.current_step<672:<1.1f} "
'''Никита, [05.11.2025 14:41]
взаимосвяз данных и вычеслений
''' 
with open('analitik.txt','w') as f:
    f.write(a)
for i in actions:
    if i==0:
        continue
    print(i,end=' ')
print()


done: False | trun: False
balans: 100 |trade: 0 | env step: 202542 | step in episode: 100



In [9]:
import h5py
f = h5py.File("D:\\model_critic_5.weights (2).h5", "r")
print(list(f.keys()))
f.close()
print(f.get('layers'))

['layers', 'vars']
None


In [5]:
import tensorflow as tf
from keras import layers as lay
import keras

In [6]:
@tf.function
def positional_encoding(seq_len, dim):
    positions = tf.range(seq_len)[:, tf.newaxis]
    dims = tf.range(dim)[tf.newaxis, :]
    angle_rates = 1 / tf.pow(10000., (2 * (dims//2)) / tf.cast(dim, tf.float32))
    angle_rads = tf.cast(positions, tf.float32) * angle_rates
    sines = tf.sin(angle_rads[:, 0::2])
    cosines = tf.cos(angle_rads[:, 1::2])
    pos_encoding = tf.concat([sines, cosines], axis=-1)
    return pos_encoding


#{'window_data':column_stak,'action_one_hot':dirat,'scalars':scalar}
input_data=lay.Input((288,10),1,name="window_data")
input_action=lay.Input((3,),1,name='action_one_hot')
input_scalar=lay.Input((6,),1,name='scalars')

#Хочешь, я покажу архитектуру PPO с Conv1D + Attention (вход — 50 свечей, выход — действие + value), с объяснением, как именно данные идут через модель?
#Это как раз то, что используют в продвинутых трейдинговых RL-моделях.
#padding="causa

x=lay.Conv1D(128,3,padding="causal",activation='relu')(input_data)
x=lay.Conv1D(128,5,padding="causal",activation='relu')(x)

pos_encoding=positional_encoding(288.,128.)




x = x + pos_encoding

x=lay.LayerNormalization()(x)
x1=lay.MultiHeadAttention(4,32,dropout=0.1)(x,x)
x=lay.Add()([x,lay.Conv1D(128,1,activation='relu')(x1)])
x=lay.LayerNormalization()(x)
x1=lay.Conv1D(128,3,activation='relu',padding='causal')(x)
x=lay.Add()([x,x1])



x = x + pos_encoding

x=lay.LayerNormalization()(x)
x1=lay.MultiHeadAttention(4,32,dropout=0.1)(x,x)
x=lay.Add()([x,lay.Conv1D(128,1,activation='relu')(x1)])
x=lay.LayerNormalization()(x)
x1=lay.Conv1D(128,3,activation='relu',padding='causal')(x)
x=lay.Add()([x,x1])


x = x + pos_encoding

x=lay.LayerNormalization()(x)
x1=lay.MultiHeadAttention(4,32,dropout=0.1)(x,x)
x=lay.Add()([x,lay.Conv1D(128,1,activation='relu')(x1)])
x=lay.LayerNormalization()(x)
x1=lay.Conv1D(128,3,activation='relu',padding='causal')(x)
x=lay.Add()([x,x1])
x=lay.LayerNormalization()(x)

x=lay.GlobalMaxPool1D()(x)


x1=lay.Dense(32,'relu')(input_action)
x2=lay.Dense(64,'relu',kernel_regularizer=keras.regularizers.L2())(input_scalar)

x2=lay.concatenate([x,x1,x2])
x2=lay.Dropout(0.15)(x2)
x=lay.Dense(256,'relu',kernel_regularizer=keras.regularizers.L2())(x2)
x=lay.Dense(64,'relu',kernel_regularizer=keras.regularizers.L2())(x)
y=lay.Dense(4)(x)

x1=lay.Dense(128,'relu',kernel_regularizer=keras.regularizers.L2())(x2)
x1=lay.Dense(32,'relu')(x1)
y1=lay.Dense(1)(x1)
actor=keras.Model([input_data,input_action,input_scalar],y)
critic=keras.Model([input_data,input_action,input_scalar],y1)
#-------------------------------------------------
#-------------------------------------------------

#actor.load_weights('model_actor_1.weights.h5')
#critic.load_weights('model_critic_1.weights.h5')

actor.summary()
critic.summary()


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ window_data         │ (1, 288, 10)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_8 (Conv1D)   │ (1, 288, 128)     │      3,968 │ window_data[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_9 (Conv1D)   │ (1, 288, 128)     │     82,048 │ conv1d_8[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_9 (Add)         │ (1, 288, 128)     │          0 │ conv1d_9[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (1, 288, 128)     │        256 │ add_9[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (1, 288, 128)     │     66,048 │ layer_normalizat… │
│ (MultiHeadAttentio… │                   │            │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_10 (Conv1D)  │ (1, 288, 128)     │     16,512 │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_10 (Add)        │ (1, 288, 128)     │          0 │ layer_normalizat… │
│                     │                   │            │ conv1d_10[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (1, 288, 128)     │        256 │ add_10[0][0]      │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_11 (Conv1D)  │ (1, 288, 128)     │     49,280 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_11 (Add)        │ (1, 288, 128)     │          0 │ layer_normalizat… │
│                     │                   │            │ conv1d_11[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_12 (Add)        │ (1, 288, 128)     │          0 │ add_11[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (1, 288, 128)     │        256 │ add_12[0][0]      │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (1, 288, 128)     │     66,048 │ layer_normalizat… │
│ (MultiHeadAttentio… │                   │            │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_12 (Conv1D)  │ (1, 288, 128)     │     16,512 │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_13 (Add)        │ (1, 288, 128)     │          0 │ layer_normalizat… │
│                     │                   │            │ conv1d_12[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (1, 288, 128)     │        256 │ add_13[0][0]      │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_13 (Conv1D)  │ (1, 288, 128)     │     49,280 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_14 (Add)        │ (1, 288, 128)     │          0 │ layer_normalizat… │
│                     │                   │            │ conv1d_13[0][0]   │
├─────────────────────┼───────────────────┼────────────┼─────────────────

 Total params: 558,212 (2.13 MB)

 Trainable params: 558,212 (2.13 MB)

 Non-trainable params: 0 (0.00 B)

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ window_data         │ (1, 288, 10)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_8 (Conv1D)   │ (1, 288, 128)     │      3,968 │ window_data[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_9 (Conv1D)   │ (1, 288, 128)     │     82,048 │ conv1d_8[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_9 (Add)         │ (1, 288, 128)     │          0 │ conv1d_9[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (1, 288, 128)     │        256 │ add_9[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (1, 288, 128)     │     66,048 │ layer_normalizat… │
│ (MultiHeadAttentio… │                   │            │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_10 (Conv1D)  │ (1, 288, 128)     │     16,512 │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_10 (Add)        │ (1, 288, 128)     │          0 │ layer_normalizat… │
│                     │                   │            │ conv1d_10[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (1, 288, 128)     │        256 │ add_10[0][0]      │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_11 (Conv1D)  │ (1, 288, 128)     │     49,280 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_11 (Add)        │ (1, 288, 128)     │          0 │ layer_normalizat… │
│                     │                   │            │ conv1d_11[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_12 (Add)        │ (1, 288, 128)     │          0 │ add_11[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (1, 288, 128)     │        256 │ add_12[0][0]      │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (1, 288, 128)     │     66,048 │ layer_normalizat… │
│ (MultiHeadAttentio… │                   │            │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_12 (Conv1D)  │ (1, 288, 128)     │     16,512 │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_13 (Add)        │ (1, 288, 128)     │          0 │ layer_normalizat… │
│                     │                   │            │ conv1d_12[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (1, 288, 128)     │        256 │ add_13[0][0]      │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_13 (Conv1D)  │ (1, 288, 128)     │     49,280 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_14 (Add)        │ (1, 288, 128)     │          0 │ layer_normalizat… │
│                     │                   │            │ conv1d_13[0][0]   │
├─────────────────────┼───────────────────┼────────────┼─────────────────

 Total params: 516,865 (1.97 MB)

 Trainable params: 516,865 (1.97 MB)

 Non-trainable params: 0 (0.00 B)